In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "volter2014cognitive")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Voelter_2014_exp1 and 3_JExpPsycholAnimLearnCogn_THES.csv")
complete_path_2 = os.path.join(original_data_pathway, "Voelter_2014_exp 2_JExpPsycholAnimLearnCogn_THES.csv")
complete_path_3 = os.path.join(original_data_pathway, "Voelter_2014_exp 4_JExpPsycholAnimLearnCogn_THES.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1 = df1.assign(experiment='1')
df2 = pd.read_csv(complete_path_2)
df2 = df2.assign(experiment='2')
df3 = pd.read_csv(complete_path_1)
df3 = df3.assign(experiment='3')
df4 = pd.read_csv(complete_path_3)
df4 = df4.assign(experiment='4')
# df4.columns


In [3]:
df4.rename(columns={"Order": "new_order"}, inplace=True)


In [4]:
data_frames=[df1, df2, df3, df4]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape"}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="volter2014cognitive"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf = fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"ape": "participant", "age":"age_in_years", "order":"order_temp","new_order":"order"}, inplace=True)

fulldf = fulldf[~fulldf.species.str.contains("gorilla")]

In [6]:
fulldf=fulldf[['study_id','experiment', 'id', 'participant', 'age_in_years','sex', 'species',
       'condition','number_trials', 'order',   'lop', 'num_levels', 'changes_direction', 
        'repetition',  't1_planning',
       't1_success', 'test_value', 'diff_num_trials',
       'order_condition', 'sum_planning_errors', 'sum_non_planning_errors',
       'sum_nonplan_error_lev1', 'non_planning_errors_level1_t1' ]]


In [7]:
for index in range(1,5):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'volter2014cognitive_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'volter2014cognitive_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [8]:
# exp1 = fulldf[fulldf['experiment'] == '1_and_3']
# exp1 = exp1.dropna(axis=1, how='all')
# exp2 = fulldf[fulldf['experiment'] == '2']
# exp2 = exp2.dropna(axis=1, how='all')
# exp3 = fulldf[fulldf['experiment'] == '4']
# exp3 = exp3.dropna(axis=1, how='all')